In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'huggingface-hub>=0.26.0', 'python-dotenv>=1.0.0',
    'pyyaml>=6.0', 'requests>=2.32.0', 'google-generativeai>=0.8.0',
], check=True)

In [ ]:
import os, json, time, threading, uuid, random, math
from pathlib import Path
from datetime import datetime
import yaml, requests
import google.generativeai as genai
import google.api_core.exceptions
from shared.gemini_rate_limiter import GeminiRateLimiter
from shared.secrets import load_secrets
from huggingface_hub import HfApi

WORK_DIR        = Path('/kaggle/working')
DUMMY_ENV_PATH  = WORK_DIR / 'dummy_env.json'
EPISODES_PATH   = WORK_DIR / 'agent_episodes.jsonl'
CHECKPOINT_PATH = WORK_DIR / 'checkpoint_p4b.json'
CONFIG_DIR      = Path('/kaggle/input/datasets/mirza176528/s2s-pipline-v2-0-2/config')

EPISODES_PER_DOMAIN = 2500
BATCH_SIZE          = 3
MAX_RETRIES         = 6
SAVE_EVERY          = 15
DOMAINS             = ['restaurant', 'banking', 'healthcare', 'education']
DIFFICULTY_DIST     = {'single_tool': 0.60, 'sequential': 0.25, 'conditional': 0.10, 'escalation': 0.05}

In [ ]:
SECRETS = load_secrets(require_gemini=True)
HF_TOKEN         = SECRETS['HF_TOKEN_PRIMARY']
GEMINI_KEY = SECRETS.get('GEMINI_API_KEY_01') or SECRETS.get('GEMINI_API_KEY_02') or ''
genai.configure(api_key=GEMINI_KEY)
GEMINI_MODEL = genai.GenerativeModel("gemini-2.5-flash")
RATE_LIMITER = GeminiRateLimiter(rpm_limit=14)

with open(CONFIG_DIR / 'hf_repos.yaml') as f:    repos_cfg     = yaml.safe_load(f)
with open(CONFIG_DIR / 'tool_registry.yaml') as f: tool_registry = yaml.safe_load(f)
with open(CONFIG_DIR / 'intent_taxonomy.yaml') as f: taxonomy   = yaml.safe_load(f)

STAGE3_REPO = repos_cfg['repos']['stage3_agent']['repo_id']
HF_API      = HfApi(token=HF_TOKEN)

if not DUMMY_ENV_PATH.exists():
    print('[dummy_env] downloading from HF...')
    for attempt in range(6):
        try:
            url = f'https://huggingface.co/datasets/{STAGE3_REPO}/resolve/main/dummy_env.json'
            r = requests.get(url, headers={'Authorization': f'Bearer {HF_TOKEN}'}, timeout=60)
            r.raise_for_status()
            with open(DUMMY_ENV_PATH, 'wb') as f: f.write(r.content)
            print('[dummy_env] downloaded'); break
        except Exception as e:
            time.sleep(min(2**attempt, 60))

with open(DUMMY_ENV_PATH, encoding='utf-8') as f:
    dummy_env = json.load(f)

print(f'[config] stage3: {STAGE3_REPO}')
print(f'[config] dummy_env domains: {[k for k in dummy_env if not k.startswith("_")]}')

In [ ]:
def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        try:
            with open(CHECKPOINT_PATH) as f: state = json.load(f)
            print(f'[checkpoint] local — generated={state["stats"]["generated"]}')
            return state
        except Exception: pass
    try:
        url = f'https://huggingface.co/datasets/{STAGE3_REPO}/resolve/main/checkpoint_p4b.json'
        r = requests.get(url, headers={'Authorization': f'Bearer {HF_TOKEN}'}, timeout=30)
        if r.status_code == 200:
            state = r.json()
            with open(CHECKPOINT_PATH, 'w') as f: json.dump(state, f)
            print(f'[checkpoint] HF fallback — generated={state["stats"]["generated"]}')
            return state
    except Exception: pass
    print('[checkpoint] fresh start')
    return {
        'by_domain': {d: 0 for d in DOMAINS},
        'stats': {'generated': 0, 'invalid': 0, 'failed': 0},
        'last_updated': None,
    }

cp_lock = threading.Lock()

def save_checkpoint(state, upload=False):
    with cp_lock:
        state['last_updated'] = datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ')
        tmp = str(CHECKPOINT_PATH) + '.tmp'
        with open(tmp, 'w') as f: json.dump(state, f)
        os.replace(tmp, str(CHECKPOINT_PATH))
    if not upload: return
    for attempt in range(6):
        try:
            HF_API.upload_file(path_or_fileobj=json.dumps(state).encode(),
                path_in_repo='checkpoint_p4b.json', repo_id=STAGE3_REPO,
                repo_type='dataset', commit_message='p4b checkpoint')
            return
        except Exception: time.sleep(min(2**attempt, 60))

state = load_checkpoint()

In [ ]:
def build_episode_system_prompt(domain):
    tools   = tool_registry.get(domain, {})
    intents = taxonomy.get(domain, []) + taxonomy.get('cross_domain', [])
    profiles_sample = random.sample(dummy_env.get(domain, []), min(3, len(dummy_env.get(domain, []))))
    return f"""You are an agent episode generation system for Pakistani Urdu customer support AI training.
Generate ONLY a valid JSON array of episode objects — no preamble, no markdown.

Domain: {domain}
Available tools and their schemas:
{json.dumps(tools, ensure_ascii=False)}

Available intents: {json.dumps(intents)}

Sample customer profiles to use as reference (use these exact values for consistency):
{json.dumps(profiles_sample, ensure_ascii=False)}

Each episode object must have:
- episode_id: string (unique)
- domain: "{domain}"
- difficulty: one of [single_tool, sequential, conditional, escalation]
- outcome: one of [success, escalated, failed]
- turns: array of turn objects
- tools_used: array of tool names actually called
- tool_chain_type: one of [single, sequential, conditional, none]
- context_extracted: object with any values pulled from user speech

Each turn must have:
- turn_id: integer from 0
- speaker: user or agent
- transcript_urdu: text in Urdu script naturally mixed with English
- intent_class: intent string for user turns, null for agent
- action_type: speak, tool_call, or generate_response
- tool: tool name string or null
- call_args: object or null — must use values consistent with customer profile
- tool_response: object or null — must be consistent with call_args
- audio_token_path: null (filled later by pipeline 5)

Critical rules:
- single_tool: exactly 1 tool call, direct resolution
- sequential: 2+ tool calls in order, each building on last
- conditional: second tool only called based on first result
- escalation: tool result cannot resolve, final turn is human handoff
- call_args values must come from the customer profile shown above
- tool_response must be internally consistent with call_args
- Urdu-English mixing must feel natural, not forced"""


def generate_episode_batch(domain, difficulty, count, system_prompt):
    user_msg = f'Generate exactly {count} {difficulty} episodes for the {domain} domain. Return only the JSON array.'
    for attempt in range(MAX_RETRIES):
        try:
            RATE_LIMITER.acquire()
            response = GEMINI_MODEL.generate_content(
                [system_prompt + "\n\n" + user_msg],
                generation_config={"response_mime_type": "application/json"}
            )
            raw = response.text.strip().replace('```json','').replace('```','').strip()
            episodes = json.loads(raw)
            if not isinstance(episodes, list): raise ValueError('Expected list')
            return episodes
        except json.JSONDecodeError:
            if attempt == MAX_RETRIES - 1: return []
            time.sleep(5 * (attempt + 1))
        except Exception as e:
            if attempt == MAX_RETRIES - 1: print(f'  [generate] failed: {e}'); return []
            time.sleep(min(2**attempt, 60))
    return []


def validate_episode(ep):
    required = ['episode_id','domain','difficulty','outcome','turns','tools_used','tool_chain_type']
    if not all(k in ep for k in required): return False
    if not isinstance(ep['turns'], list) or len(ep['turns']) < 2: return False
    if not any(t.get('action_type') == 'tool_call' for t in ep['turns']): return False
    for t in ep['turns']:
        if t.get('action_type') == 'tool_call':
            if not t.get('tool') or not t.get('tool_response'): return False
    return True

In [ ]:
for domain in DOMAINS:
    already_done = state['by_domain'].get(domain, 0)
    remaining    = EPISODES_PER_DOMAIN - already_done

    if remaining <= 0:
        print(f'[{domain}] complete ({already_done}/{EPISODES_PER_DOMAIN})')
        continue

    print(f'\n[{domain}] generating {remaining} episodes ({already_done} done)')

    diff_counts = {
        d: math.ceil(remaining * w)
        for d, w in DIFFICULTY_DIST.items()
    }

    for difficulty, total_for_diff in diff_counts.items():
        generated = 0
        batch_num = 0

        while generated < total_for_diff:
            system_prompt = build_episode_system_prompt(domain)
            ask_count     = min(BATCH_SIZE, total_for_diff - generated)
            episodes      = generate_episode_batch(domain, difficulty, ask_count, system_prompt)
            batch_num    += 1
            valid_count   = 0

            for ep in episodes:
                if not validate_episode(ep):
                    with cp_lock: state['stats']['invalid'] += 1
                    continue
                ep['episode_id']   = f'{domain}_{difficulty}_{str(uuid.uuid4())[:8]}'
                ep['generated_at'] = datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ')
                ep['split']        = 'train'
                with open(EPISODES_PATH, 'a', encoding='utf-8') as f:
                    f.write(json.dumps(ep, ensure_ascii=False) + '\n')
                valid_count += 1
                generated   += 1
                with cp_lock:
                    state['stats']['generated']  += 1
                    state['by_domain'][domain]   += 1

            if not episodes:
                with cp_lock: state['stats']['failed'] += 1

            if batch_num % SAVE_EVERY == 0:
                save_checkpoint(state, upload=True)

            print(f'  [{domain}/{difficulty}] batch {batch_num} valid={valid_count}/{len(episodes)} total={generated}/{total_for_diff}')
            time.sleep(1)

    save_checkpoint(state, upload=True)

total_eps = sum(1 for _ in open(EPISODES_PATH, encoding='utf-8')) if EPISODES_PATH.exists() else 0
print(f'\n[p4b] {total_eps} episodes written')
print('[done] ready for p4c_upload.ipynb')